# BPA Master Workflow: Pattern Classification (Task 1) & TSD Estimation (Task 2)
**Toronto Metropolitan University — Major Research Project (MRP)**

---

## 🩸 Project Overview & Scientific Workflow
This master notebook coordinates the end-to-end execution of both core tasks in the Major Research Project:
1. **Task 1: Multi-Task Bloodstain Pattern Classification**: Predicts pattern categories (excluding Cough Spatter to prevent leakage) and physical force mechanisms, leading to physics-based Area of Origin calculations.
2. **Task 2: Time Since Deposition (TSD) Estimation**: Predicts temporal age splits (*Fresh*, *Intermediate*, *Aged*) based on hemoglobin oxidation color-space shifts over a 28-day drying cycle.

---

## ⚙️ Phase 1: Environment Setup & GPU Verification
Installs required libraries and checks if a high-speed GPU (T4 on Google Colab) is active.

In [ ]:
# Check GPU acceleration
import torch
print('\n=== GPU Device Check ===')
if torch.cuda.is_available():
    print(f'[+] GPU Active: {torch.cuda.get_device_name(0)}')
else:
    print('[!] GPU not active. Running training from scratch is highly recommended with GPU support.')

In [ ]:
# Install required dependencies
!pip install -q timm scikit-learn seaborn matplotlib python-docx albumentations jinja2
print('[+] Dependency libraries installed successfully!')

--- 
## 🎯 Option A: Run Inference Using Saved Models
If you want to immediately run evaluations or test individual images using the pre-trained weights already saved in the repository, use this section. No training from scratch is required.

### 📂 Step A.1: Navigate to Repository Root
To run imports, load weight files, and access datasets, Colab needs to be pointing to the repository's root folder. Choose one of the options below:

#### 🌐 Option 1: Clone directly to Colab Local VM (Recommended for Reviewers)
This clones the code repository directly from GitHub into Colab's high-speed local VM storage. Simply run the setup cell below without doing anything.

#### 💾 Option 2: Mount Personal Google Drive (Recommended for persistent development)
1. Open the 👉 **[Shared Google Drive Archive Link](https://drive.google.com/drive/folders/1Qg8CjBzJ2wPJIfGgGEfSezEaNndCxuGy?dmr=1&ec=wgc-drive-%5Bmodule%5D-goto)**
2. Click the folder name at the top, select **Organize** $\rightarrow$ **Add shortcut**, and choose **My Drive**.
3. Run the code cell below to automatically search and mount your Drive.

In [ ]:
import os
import sys

# =====================================================================
# DIRECT PATH SETUP CELL
# This script mounts Google Drive and sets the working directory directly
# to your project folder. If the folder is not found, it clones from GitHub.
# =====================================================================

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    pass

project_path = None

if IN_COLAB:
    # 1. Mount Google Drive
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print(f"[!] Drive mount failed: {e}")
    
    # 2. Direct path in Google Drive
    exact_path = '/content/drive/MyDrive/A-Hybrid-Framework-for-Forensic-Bloodstain-Pattern-Analysis'
    
    if os.path.exists(exact_path) and os.path.exists(os.path.join(exact_path, 'Task_1_Classification')):
        project_path = exact_path
            
    # 3. Fallback: Clone from GitHub if shortcut was not added to Drive
    if not project_path:
        print("[!] Project folder not found in Drive. Cloning from GitHub instead...")
        from IPython import get_ipython
        ipython = get_ipython()
        if not os.path.exists('A-Hybrid-Framework-for-Forensic-Bloodstain-Pattern-Analysis'):
            ipython.system('git clone https://github.com/shahidabatool/A-Hybrid-Framework-for-Forensic-Bloodstain-Pattern-Analysis.git')
        
        fallback_path = '/content/A-Hybrid-Framework-for-Forensic-Bloodstain-Pattern-Analysis'
        if os.path.exists(fallback_path):
            project_path = fallback_path
else:
    # Local Jupyter/Terminal environment
    project_path = os.getcwd()

if project_path:
    # Set working directory dynamically for Python and shell
    from IPython import get_ipython
    get_ipython().run_line_magic('cd', project_path)
    print(f"\n[+] Working directory successfully set to:\n    {os.getcwd()}")
else:
    print("[!] Error: Could not resolve working directory. Please set it manually.")


### ⚠️ Notice: Pre-trained Weights Download
Because deep learning weight checkpoints (`.pth` files) are large and exceed GitHub's 100MB file limit, they are excluded from the git push commits.

- **If using Git Clone fallback**: Please download the `.pth` weights from the [Shared Google Drive Archive](https://drive.google.com/drive/folders/1Qg8CjBzJ2wPJIfGgGEfSezEaNndCxuGy?dmr=1&ec=wgc-drive-%5Bmodule%5D-goto) and upload them manually into the respective local Colab folders (`Task_1_Classification/Models/` and `Task_2_TSD/Models/`).
- **If using Google Drive Mount**: Because you added a shortcut to your Drive, all weights and datasets inside the folder are already loaded and ready in your directory! No downloads or uploads are required.

### Step A.2: Import Architectures & Load Weights
This dynamically appends the correct model directories relative to your current workspace root folder.

In [ ]:
import sys
import torch
import torchvision.transforms as transforms
from PIL import Image
import os

# Dynamically append import folders relative to current working directory
sys.path.append(os.path.join(os.getcwd(), 'Task_1_Classification/Code/training'))
sys.path.append(os.path.join(os.getcwd(), 'Task_2_TSD/Code/models'))

from train_efficientnet import BPAMultiTaskEfficientNet
from bloodnet import bloodnet50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[+] Using device: {device}')

# 1. Load Task 1 Classifier Model (4-class pattern setup)
t1_model = BPAMultiTaskEfficientNet(num_pattern=4, num_mechanism=3)
t1_weights_path = 'Task_1_Classification/Models/model1_efficientnet_b0.pth'
if os.path.exists(t1_weights_path):
    t1_model.load_state_dict(torch.load(t1_weights_path, map_location=device))
    t1_model.to(device).eval()
    print('[+] Task 1 (EfficientNet-B0) weights loaded successfully!')
else:
    print(f'[!] Warning: Weights file not found at {t1_weights_path}')

# 2. Load Task 2 TSD Model
t2_model = bloodnet50(num_classes=3)
t2_weights_path = 'Task_2_TSD/Models/best_tsd_model_resnet50.pth'
if os.path.exists(t2_weights_path):
    t2_model.load_state_dict(torch.load(t2_weights_path, map_location=device))
    t2_model.to(device).eval()
    print('[+] Task 2 (ResNet-50 CBAM) weights loaded successfully!')
else:
    print(f'[!] Warning: Weights file not found at {t2_weights_path}')

### 📊 Step A.2.b: Visualize Pre-trained Model Evaluation Metrics
This cell loads and displays the pre-generated evaluation plots (Confusion Matrices and ROC-AUC Curves) for both the Task 1 (EfficientNet-B0) and Task 2 (ResNet-50 CBAM) models. This allows you to verify the models' validation performance immediately without training them.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# Setup the paths to the pre-generated evaluation plots
t1_cm_path = 'Task_1_Classification/Evaluation/efficientnet_pattern_confusion_matrix.png'
t1_roc_path = 'Task_1_Classification/Evaluation/efficientnet_pattern_roc_auc.png'
t2_cm_path = 'Task_2_TSD/Evaluation/resnet50_tsd_confusion_matrix.png'
t2_roc_path = 'Task_2_TSD/Evaluation/resnet50_tsd_roc_auc.png'

paths = [t1_cm_path, t1_roc_path, t2_cm_path, t2_roc_path]
existing_paths = [p for p in paths if os.path.exists(p)]

if len(existing_paths) == 4:
    print("[+] Loading evaluation metrics and curves...")
    fig, axes = plt.subplots(2, 2, figsize=(15, 13))
    
    # Task 1 - Confusion Matrix
    axes[0, 0].imshow(mpimg.imread(t1_cm_path))
    axes[0, 0].set_title("Task 1: Pattern Classification Confusion Matrix", fontsize=11, fontweight='bold')
    axes[0, 0].axis('off')
    
    # Task 1 - ROC Curve
    axes[0, 1].imshow(mpimg.imread(t1_roc_path))
    axes[0, 1].set_title("Task 1: Pattern Classification ROC-AUC Curve", fontsize=11, fontweight='bold')
    axes[0, 1].axis('off')
    
    # Task 2 - Confusion Matrix
    axes[1, 0].imshow(mpimg.imread(t2_cm_path))
    axes[1, 0].set_title("Task 2: TSD Age Confusion Matrix", fontsize=11, fontweight='bold')
    axes[1, 0].axis('off')
    
    # Task 2 - ROC Curve
    axes[1, 1].imshow(mpimg.imread(t2_roc_path))
    axes[1, 1].set_title("Task 2: TSD Age ROC-AUC Curve", fontsize=11, fontweight='bold')
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("[!] Could not load all evaluation plots. Running diagnostic check on paths:")
    for p in paths:
        print(f" - {p}: {'FOUND' if os.path.exists(p) else 'MISSING'}")

### Step A.3: Run Unified Inference Example
Define the unified forecast workflow. This method automatically handles dictionary output unpacking to prevent errors.

In [ ]:
from IPython.display import display
import torch
from PIL import Image
import os

def predict_bloodstain(image_path):
    if not os.path.exists(image_path):
        print(f'[!] File not found: {image_path}')
        print("Please verify the image path or try pointing to another image.")
        return
        
    # Preprocessing transforms
    t1_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    t2_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Load and display the image
    img = Image.open(image_path).convert('RGB')
    print("📸 Input Bloodstain Image:")
    display(img.resize((256, 256)))
    
    # Task 1 Classification
    img_t1 = t1_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        t1_out = t1_model(img_t1)
        pattern_out = t1_out['pattern']
        mech_out = t1_out['mechanism']
        
        pattern_idx = torch.argmax(pattern_out, dim=1).item()
        mech_idx = torch.argmax(mech_out, dim=1).item()
        
    # Sourced active 4-class labels matching training output layer
    t1_pattern_labels = ['Gunshot', 'Impact Spatter', 'Passive Drip', 'Transfer/Wipe']
    t1_mech_labels = ['Passive', 'Low Velocity', 'Medium/High Velocity']
    
    # Task 2 Temporal Age Prediction
    img_t2 = t2_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        age_out = t2_model(img_t2)
        age_idx = torch.argmax(age_out, dim=1).item()
        
    t2_age_labels = ['Fresh (1 Day)', 'Intermediate (7-14 Days)', 'Aged (21-28 Days)']
    
    print('\n==================================================')
    print(f'🩸 FORENSIC ANALYSIS REPORT FOR: {os.path.basename(image_path)}')
    print('==================================================')
    print(f'[+] Predicted Pattern:    {t1_pattern_labels[pattern_idx]}')
    print(f'[+] Predicted Mechanism:  {t1_mech_labels[mech_idx]}')
    print(f'[+] Estimated Age (TSD):  {t2_age_labels[age_idx]}')
    print('==================================================')

### 🎯 Step A.3.b: Run Inference on Your Uploaded Image
This cell executes the prediction pipeline on your custom uploaded image. By default, it searches for a file named `sample.jpeg` in your current working directory or the root `/content` directory.

In [ ]:
import os

# Smart path check for sample.jpeg
possible_paths = [
    'sample.jpeg',
    'sample.jpg',
    '/content/sample.jpeg',
    '/content/sample.jpg'
]

test_image_path = None
for path in possible_paths:
    if os.path.exists(path):
        test_image_path = path
        break

if test_image_path:
    print(f"[+] Running predictions on: {test_image_path}\n")
    predict_bloodstain(test_image_path)
else:
    print("[!] Error: Could not locate 'sample.jpeg' anywhere.")
    print("Please upload your image to the left Colab files sidebar and name it exactly 'sample.jpeg'.")

--- 
## ⚙️ Option B: Re-run Training & Evaluation Pipelines
This section runs model training and evaluations directly using your Google Drive project folders (no ZIP files required).

### Step B.1: Run Parent-Image Leakage Audit (Task 1)

In [ ]:
!python Task_1_Classification/Code/preprocessing/audit_data_leakage.py

### Step B.2: Train Task 1 Convolutional Models

In [ ]:
print('=== Training Multi-Task EfficientNet-B0 ===')
!python Task_1_Classification/Code/training/train_efficientnet.py \
  --data_dir Task_1_Classification/Data/Augmented/train \
  --val_dir Task_1_Classification/Data/Augmented/val \
  --epochs 10 \
  --batch_size 32

print('\n=== Training Multi-Task ResNet-50 ===')
!python Task_1_Classification/Code/training/train_resnet50.py \
  --data_dir Task_1_Classification/Data/Augmented/train \
  --val_dir Task_1_Classification/Data/Augmented/val \
  --epochs 10 \
  --batch_size 32

print('\n=== Training Multi-Task ConvNeXt-Tiny ===')
!python Task_1_Classification/Code/training/train_convnext.py

### Step B.3: Evaluate Task 1 Classifiers

In [ ]:
print('=== Evaluating EfficientNet-B0 ===')
!python Task_1_Classification/Code/evaluation/evaluate_efficientnet.py \
  --test_dir Task_1_Classification/Data/Augmented/test

print('\n=== Evaluating ResNet-50 ===')
!python Task_1_Classification/Code/evaluation/evaluate_resnet50.py \
  --test_dir Task_1_Classification/Data/Augmented/test

print('\n=== Evaluating ConvNeXt-Tiny ===')
!python Task_1_Classification/Code/evaluation/evaluate_convnext.py

### Step B.4: Train Task 2 (TSD) Models

In [ ]:
print('=== Training Fine-Tuned ResNet-50 TSD ===')
!python Task_2_TSD/Code/models/train_tsd_resnet50.py \
  --data_dir Task_2_TSD/Text/BloodNet_50k_Images/data/train1 \
  --val_dir Task_2_TSD/Text/BloodNet_50k_Images/data/test \
  --weights_path Task_2_TSD/Text/BloodNet_50k_Images/bloodnet50_new.pth \
  --output_dir Task_2_TSD/Models \
  --epochs 10 \
  --batch_size 128 \
  --lr 1e-4

print('\n=== Training EfficientNet-B0 TSD ===')
!python Task_2_TSD/Code/models/train_tsd_efficientnet.py \
  --data_dir Task_2_TSD/Text/BloodNet_50k_Images/data/train1 \
  --val_dir Task_2_TSD/Text/BloodNet_50k_Images/data/test \
  --output_dir Task_2_TSD/Models \
  --epochs 10 \
  --batch_size 128 \
  --lr 1e-4

print('\n=== Training ConvNeXt-Tiny TSD ===')
!python Task_2_TSD/Code/models/train_tsd_convnext.py \
  --data_dir Task_2_TSD/Text/BloodNet_50k_Images/data/train1 \
  --val_dir Task_2_TSD/Text/BloodNet_50k_Images/data/test \
  --output_dir Task_2_TSD/Models \
  --epochs 10 \
  --batch_size 128 \
  --lr 1e-4

### Step B.5: Evaluate Task 2 TSD Backbones

In [ ]:
!python Task_2_TSD/Code/evaluation/evaluate_tsd.py \
  --test_dir Task_2_TSD/Text/BloodNet_50k_Images/data/outside_test \
  --baseline_weights Task_2_TSD/Text/BloodNet_50k_Images/bloodnet50_new.pth \
  --resnet50_weights Task_2_TSD/Models/best_tsd_model_resnet50.pth \
  --efficientnet_weights Task_2_TSD/Models/best_tsd_model_efficientnet.pth \
  --convnext_weights Task_2_TSD/Models/best_tsd_model_convnext.pth \
  --output_dir Task_2_TSD/Evaluation

### Step B.6: Generate Word Reports

In [ ]:
!python Task_1_Classification/Code/generate_word_report.py
!python Task_2_TSD/Code/generate_tsd_word_report.py
print('[+] Word reports generated successfully!')

---
## 🖥️ Streamlit Dashboard & Local Execution
For an interactive, user-friendly visualization of the decision support system, you can run the Streamlit dashboard locally on your machine. 

We have provided a double-clickable launch file: **`run_dashboard.command`** in the repository root.

### How to run the dashboard:
1. Locate **`run_dashboard.command`** in your project folder on your computer.
2. On **macOS**, simply **double-click** the file. It will automatically launch the terminal, activate the `mrp_env` virtual environment, and open the interactive Streamlit dashboard in your default browser.
3. On **Windows/Linux**, open your terminal in the project directory and run:
   ```bash
   source mrp_env/bin/activate
   streamlit run dashboard.py
   ```